# Fleet v0.9 — Dynamic Experts

This notebook builds the serious bank, rebuilds the v0.9 dynamic semantic ABI, creates the standalone `FleetForCausalLM`, verifies install/uninstall, and can publish it.

Use a GPU runtime. Put a **rotated** Hugging Face write token in Colab Secrets as `HF_TOKEN` if you want the final upload.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!rm -rf /content/fleet-v0.9
!mkdir -p /content/fleet-v0.9
!unzip -q /content/fleet-semantic-v0.9-dynamic.zip -d /content/fleet-v0.9
%cd /content/fleet-v0.9
!pip install -q -U pip
!pip install -q -e ".[server,dev]"

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 1. Compile the serious expert bank

In [ ]:
!fleet build \
  --config configs/experts.json \
  --output artifacts/fleet-bank

## 2. Rebuild v0.9 semantic space

Do not reuse an old v0.8 `semantic_router.safetensors`. v0.9 uses NULL-anchor-only centering and per-expert thresholds so prototypes can be hot-plugged.

In [ ]:
!rm -f artifacts/fleet-bank/semantic_router.json artifacts/fleet-bank/semantic_router.safetensors
!fleet semantic-build \
  --bank artifacts/fleet-bank \
  --anchors configs/semantic_anchors.json

In [ ]:
!fleet validate artifacts/fleet-bank

## 3. Create a portable FleetAdapter package

In [ ]:
import json
bank = json.load(open("artifacts/fleet-bank/fleet_bank.json"))
available_experts = [x["name"] for x in bank["experts"]]
print("Available experts:", available_experts)
if not available_experts:
    raise RuntimeError("The compiled bank contains no experts.")
EXPERT = available_experts[0]
assert EXPERT.strip(), "Selected expert name is blank."
print("Packaging:", EXPERT)

In [ ]:
!mkdir -p dist
!fleet adapter-pack \
  --bank artifacts/fleet-bank \
  --expert "{EXPERT}" \
  --output "dist/{EXPERT}.fleet"

In [ ]:
!fleet adapter-inspect "dist/{EXPERT}.fleet"

## 4. Build the standalone native model

In [ ]:
!fleet native-build \
  --bank artifacts/fleet-bank \
  --output artifacts/Fleet-Semantic-4B \
  --dtype float16 \
  --max-shard-size 4GB

## 5. Mandatory dynamic round-trip validation

This test automatically exports one installed expert, uninstalls it, verifies the semantic bank shrank, reinstalls it, verifies the original expert order returned, then generates a probe.

In [ ]:
!fleet native-validate --model artifacts/Fleet-Semantic-4B

## 6. Direct API demo

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "artifacts/Fleet-Semantic-4B",
    trust_remote_code=True,
    dtype=torch.float16,
    device_map="auto",
)

model.experts()

In [ ]:
# Hot remove + re-install the portable package we built above.
name = EXPERT
package = f"dist/{name}.fleet"

# If this expert is already installed, remove it first.
if name in [x["name"] for x in model.experts()]:
    print("removed:", model.uninstall(name))

print("after uninstall:", [x["name"] for x in model.experts()])
print("installed:", model.install(package))
print("after install:", [x["name"] for x in model.experts()])

You can also install a published package directly:

```python
model.install("hf://noumenon-labs/Fleet-Adapters/robotics.fleet")
```

## 7. Automatic Hub release

This rebuilds and re-validates before upload. It will not upload if the native dynamic round-trip fails.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!fleet native-release \
  --bank artifacts/fleet-bank \
  --output artifacts/Fleet-Semantic-4B \
  --repo noumenon-labs/Fleet-Semantic-4B \
  --max-shard-size 4GB \
  --num-workers 4